<a href="https://colab.research.google.com/github/swatijaincts/GENAI-Course/blob/Colab-link/Legal_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Legal RAG **
Section-Aware Retrieval-Augmented Generation for Statutory Law PDFs

A Retrieval-Augmented Generation (RAG) pipeline for legal statutes requires structure-aware chunking and metadata filtering so that individual sections, sub-clauses, and provisos are never split mid-rule or confused across different acts.

End-to-End RAG Architecture for Legal PDFs

The pipeline splits into two distinct phases: Offline Ingestion (processing the statute PDFs into an indexed knowledge base) and Online Retrieval & Generation (answering user queries with exact statutory citations).

# Library Installation

In [ ]:
!pip install -q pymupdf sentence-transformers rank_bm25 langchain-text-splitters

# Data extraction from PDF
Bare Act PDFs (including Bharatiya Nyaya Sanhita (BNS), 2023, Bharatiya Nagarik Suraksha Sanhita (BNSS), 2023, Bharatiya Sakshya Adhiniyam (BSA), 2023,

In [ ]:
import fitz  # PyMuPDF
import re
import os

def extract_sections_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    act_name = os.path.splitext(os.path.basename(pdf_path))[0].replace('_', ' ')
    chapters = []
    current_chapter = "Unknown Chapter"

    # Basic structural parsing using regular expressions
    full_text = ""
    for page in doc:
        full_text += page.get_text("text") + "\n"

    # Split into sections based on typical legislative formats (e.g., 'Section 1. ', '1. Short title...')
    raw_sections = re.split(r'\n(?=\d+\.\s+[A-Z])', full_text)

    chunks = []
    for idx, raw_sec in enumerate(raw_sections):
        if idx == 0:
            continue # Skip introductory content or title page

        lines = raw_sec.strip().split('\n')
        header = lines[0]
        body = "\n".join(lines[1:])

        # Extract section number
        sec_num_match = re.match(r'^(\d+)\.', header)
        section_number = sec_num_match.group(1) if sec_num_match else str(idx)

        metadata = {
            "act_name": act_name,
            "section_number": section_number,
            "header": header,
            "jurisdiction": "India",
            "status": "active"
        }

        chunks.append({
            "text": f"{act_name} - {header}\n\n{body}",
            "metadata": metadata
        })

    return chunks

# Test the parser with one of the available PDFs
pdf_files = [
    "/content/Bharatiya Sakshya Adhiniyam_2023.pdf",
    "/content/Bharatiya Nagarik Suraksha Sanhita_2023.pdf",
    "/content/Bharatiya Nyaya Sanhita_2023.pdf"
]

parsed_documents = []
if os.path.exists(pdf_files[0]):
    parsed_documents = extract_sections_from_pdf(pdf_files[0])
    print(f"Successfully parsed {len(parsed_documents)} sections from {pdf_files[0]}.")
else:
    print("PDF file not found for processing.")

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


Successfully parsed 345 sections from /content/Bharatiya Sakshya Adhiniyam_2023.pdf.


# Dense Semantic Indexing & Embedding

In [ ]:
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import numpy as np

# 1. Initialize the dense encoder
print("Loading embedding model...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Prepare documents for indexing
doc_texts = [doc['text'] for doc in parsed_documents]
doc_metadata = [doc['metadata'] for doc in parsed_documents]

# 3. Generate dense embeddings
print("Generating dense vector embeddings...")
dense_embeddings = embedding_model.encode(doc_texts, show_progress_bar=True)

# 4. Generate sparse BM25 index
print("Building BM25 sparse index...")
tokenized_corpus = [text.lower().split() for text in doc_texts]
bm25_index = BM25Okapi(tokenized_corpus)

print(f"Vector database indexing completed. Built {len(dense_embeddings)} dense and sparse indices.")

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (7 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating dense vector embeddings...


Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Building BM25 sparse index...
Vector database indexing completed. Built 345 dense and sparse indices.


# Encoding-Cross Encoder

In [ ]:
from sentence_transformers import CrossEncoder
import numpy as np
import re

# 1. Load a Cross-Encoder for Re-ranking
print("Loading Cross-Encoder model...")
reranker_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def hybrid_search(query, top_k=5, dense_weight=0.7):
    """
    Performs hybrid search combining dense (vector) embeddings and sparse (BM25) search
    with multi-field metadata filtering (section_number and act_name).
    """
    # 1. Parse Metadata Filters
    sec_filter = None
    act_filter = None

    # Extract section filter (e.g., "section 3") without syntax error splits
    sec_match = re.search(r'section\s+(\d+)', query.lower())
    if sec_match:
        sec_filter = sec_match.group(1)
        print(f"Extracted metadata filter: section_number = '{sec_filter}'")

    # Extract act_name filter using common statutory nicknames / keywords
    query_lower = query.lower()
    if "sakshya" in query_lower or "adhiniyam" in query_lower or "bsa" in query_lower:
        act_filter = "Bharatiya Sakshya Adhiniyam"
    elif "nagarik" in query_lower or "suraksha" in query_lower or "bnss" in query_lower:
        act_filter = "Bharatiya Nagarik Suraksha Sanhita"
    elif "nyaya" in query_lower or "sanhita" in query_lower or "bns" in query_lower:
        act_filter = "Bharatiya Nyaya Sanhita"

    if act_filter:
        print(f"Extracted metadata filter: act_name = '{act_filter}'")

    # 2. Score Computation
    # Dense Search scores
    query_emb = embedding_model.encode(query)
    norms = np.linalg.norm(dense_embeddings, axis=1)
    query_norm = np.linalg.norm(query_emb)
    dense_scores = np.dot(dense_embeddings, query_emb) / (norms * query_norm + 1e-9)

    # Sparse Search scores
    tokenized_query = query.lower().split()
    bm25_scores = np.array(bm25_index.get_scores(tokenized_query))
    if bm25_scores.max() > 0:
        bm25_scores = bm25_scores / bm25_scores.max()

    # Combined hybrid scores
    sparse_weight = 1.0 - dense_weight
    hybrid_scores = (dense_weight * dense_scores) + (sparse_weight * bm25_scores)

    # 3. Multi-field Metadata Filtering
    matched_indices = []
    for idx, doc in enumerate(parsed_documents):
        metadata = doc['metadata']

        # Check section number match
        if sec_filter and metadata.get('section_number') != sec_filter:
            continue

        # Check act name match (using substring search)
        if act_filter and act_filter.lower() not in metadata.get('act_name', '').lower():
            continue

        matched_indices.append(idx)

    # Fallback to unrestricted search if metadata filters return no matches
    if not matched_indices:
        print("No documents matched filters. Falling back to unrestricted search.")
        matched_indices = list(range(len(parsed_documents)))

    # 4. Hybrid Ranking & Cross-Encoder Re-Ranking
    ranked_subset = sorted(matched_indices, key=lambda idx: hybrid_scores[idx], reverse=True)
    top_candidates = ranked_subset[:20]

    pairs = [[query, parsed_documents[idx]['text']] for idx in top_candidates]
    rerank_scores = reranker_model.predict(pairs)

    reranked_results = [parsed_documents[top_candidates[i]] for i in np.argsort(rerank_scores)[::-1]]
    return reranked_results[:top_k]

# Test with query referencing a specific act and section
test_query = "What does section 3 of Sakshya Adhiniyam tell us about evidence?"
retrieved_docs = hybrid_search(test_query, top_k=2, dense_weight=0.6)

print(f"\nTop Retrieved Document:\n{retrieved_docs[0]['text'][:300]}...")

Loading Cross-Encoder model...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Extracted metadata filter: section_number = '3'
Extracted metadata filter: act_name = 'Bharatiya Sakshya Adhiniyam'

Top Retrieved Document:
Bharatiya Sakshya Adhiniyam 2023 - 3. Evidence may be given of facts in issue and relevant facts.— Evidence may be given in any suit 

or proceeding of the existence or non-existence of every fact in issue and of such other facts as are 
hereinafter declared to be relevant, and of no others. 
Explan...


### Online Generation Phase: LLM Answer Synthesis with Gemini
This step integrates Google Gemini to synthesize a structured, legally grounded answer using only the retrieved statutory context. We use the updated `google-genai` SDK and enforce citation grounding.

In [ ]:
import getpass
from google import genai
from google.colab import userdata

def generate_answer(query, retrieved_documents):
    """
    Generates a context-grounded response using the modern google-genai library
    based on retrieved statutory documents.
    """
    # 1. Retrieve the secure API Key from Colab Secrets or prompt user
    api_key = None
    try:
        api_key = userdata.get('GOOGLE_API_KEY')
    except Exception:
        pass

    if not api_key:
        print("GOOGLE_API_KEY secret not found in Colab Secrets.")
        print("Please enter your Gemini API key below to continue (it will not be saved permanently):\n")
        api_key = getpass.getpass("Gemini API Key: ")

    if not api_key or api_key.strip() == "":
        return "Error: No API key provided. Please configure GOOGLE_API_KEY."

    try:
        # Correct client initialization for the modern google-genai SDK
        client = genai.Client(api_key=api_key)
    except Exception as e:
        return f"Error initializing modern Gemini client: {e}"

    # 2. Construct context from retrieved documents
    context_blocks = []
    for idx, doc in enumerate(retrieved_documents):
        meta = doc.get('metadata', {})
        act = meta.get('act_name', 'Unknown Act')
        sec = meta.get('section_number', 'Unknown Section')
        context_blocks.append(f"Source [{idx + 1}]: {act} (Section {sec})\n{doc['text']}")

    context = "\n\n---\n\n".join(context_blocks)

    # 3. Create instruction-tuned legal prompt
    prompt = f"""You are an expert legal assistant specializing in Indian statutory law.
Your task is to answer the User Query using ONLY the provided Statutory Context.

Strict Grounding Instructions:
- Base your answer solely on the provided Statutory Context.
- Cite the specific Section and Act (e.g., "Section 3 of Bharatiya Sakshya Adhiniyam 2023") when stating any rule.
- If the context does not contain enough information to answer, state clearly that the answer is not available in the provided sources.

Statutory Context:
{context}

User Query: {query}

Legal Response:"""

    # 4. Invoke modern model with fallback cascade utilizing active available endpoints
    candidate_models = ['gemini-2.5', 'gemini-1.5-flash']
    last_err = None
    for model_name in candidate_models:
        try:
            response = client.models.generate_content(
                model=model_name,
                contents=prompt
            )
            return response.text
        except Exception as e:
            last_err = e
            continue

    return f"Error generating content from Gemini API (tried fallback models): {last_err}"

# Run generation test using previously retrieved documents
answer = generate_answer(test_query, retrieved_docs)
print("=== Synthesized Answer ===\n")
print(answer)

GOOGLE_API_KEY secret not found in Colab Secrets.
Please enter your Gemini API key below to continue (it will not be saved permanently):

Gemini API Key: ··········
=== Synthesized Answer ===

Error generating content from Gemini API (tried fallback models): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-pro is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}


### Alternative: Local LLM Synthesis using Ollama

To use Ollama in Google Colab, we need to:
1. Download and run the Ollama binary as a background process.
2. Pull a lightweight model (such as `gemma2:2b` or `llama3` which run efficiently on CPU or standard GPU runtimes).
3. Use the Python `requests` library or the `ollama` client to generate structured, grounded legal answers.

In [ ]:
# 1. Install Ollama in the Colab system
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Start the Ollama background server
import subprocess
import time
import requests

print("Starting Ollama background process...")
process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Wait for the server to spin up
time.sleep(5)

# 3. Pull a fast, lightweight legal-capable model
print("Pulling gemma2:2b (lightweight and optimized for instruction-following)...")
subprocess.run(["ollama", "pull", "gemma2:2b"])

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:3 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [358 B]
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Get:6 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [81.1 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [3,143 kB]
Get:10 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:11 http://archive.ubuntu.com/ubuntu noble-updates/restricted amd64 Packages [2,329 kB]
Get:12 http://security.ubuntu.com/ubuntu noble-security/restricted amd64 Packages [2,211 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.6 MB]
Get:14

CompletedProcess(args=['ollama', 'pull', 'gemma2:2b'], returncode=0)

In [ ]:
import json
import requests

def generate_answer_ollama(query, retrieved_documents, model_name="gemma2:2b"):
    """
    Generates a context-grounded response using a local Ollama instance.
    """
    # 1. Construct context from retrieved documents
    context_blocks = []
    for idx, doc in enumerate(retrieved_documents):
        meta = doc.get('metadata', {})
        act = meta.get('act_name', 'Unknown Act')
        sec = meta.get('section_number', 'Unknown Section')
        context_blocks.append(f"Source [{idx + 1}]: {act} (Section {sec})\n{doc['text']}")

    context = "\n\n---\n\n".join(context_blocks)

    # 2. Create instruction-tuned legal prompt
    prompt = f"""You are an expert legal assistant specializing in Indian statutory law.
Your task is to answer the User Query using ONLY the provided Statutory Context.

Strict Grounding Instructions:
- Base your answer solely on the provided Statutory Context.
- Cite the specific Section and Act (e.g., "Section 3 of Bharatiya Sakshya Adhiniyam 2023") when stating any rule.
- If the context does not contain enough information to answer, state clearly that the answer is not available in the provided sources.

Statutory Context:
{context}

User Query: {query}

Legal Response:"""

    # 3. Query the local Ollama API endpoint
    url = "http://localhost:11434/api/generate"
    payload = {
        "model": model_name,
        "prompt": prompt,
        "stream": False
    }

    try:
        # Set timeout to None to allow the local compilation and generation to complete without interrupting
        response = requests.post(url, json=payload, timeout=None)
        if response.status_code == 200:
            return response.json().get("response", "Error: Empty response.")
        else:
            return f"Ollama returned error: {response.status_code} - {response.text}"
    except Exception as e:
        return f"Could not connect to local Ollama server. Ensure it is running. Error: {e}"

# Run generation test using local Ollama model
print("Synthesizing answer using local Ollama (gemma2:2b)...\n")
answer_ollama = generate_answer_ollama(test_query, retrieved_docs)
print("=== Synthesized Answer (Ollama) ===\n")
print(answer_ollama)

Synthesizing answer using local Ollama (gemma2:2b)...

=== Synthesized Answer (Ollama) ===

Section 3 of the Bharatiya Sakshya Adhiniyam 2023 states that "Evidence may be given of facts in issue and relevant facts."  It also further states, "This section shall not enable any person to give evidence of a fact which he is disentitled to prove by any provision of the law for the time being in force relating to civil procedure". 



### Phase 4: Unified Legal RAG Interface

This cell bundles the parsing, hybrid search, metadata filtering, cross-encoder re-ranking, and local Ollama generation into a single end-to-end function: `ask_legal_assistant(query)`.

In [ ]:
def ask_legal_assistant(query, top_k=3):
    """
    End-to-End unified function to retrieve relevant clauses and synthesize an answer.
    """
    print(f"[*] Processing query: '{query}'")

    # 1. Retrieve the top relevant sections
    print("[*] Retrieving relevant statutory context...")
    retrieved_docs = hybrid_search(query, top_k=top_k)

    if not retrieved_docs:
        return "No relevant legal documents could be retrieved to answer this query."

    # 2. Synthesize grounded answer via local Ollama
    print("[*] Synthesizing context-grounded response via Ollama...")
    answer = generate_answer_ollama(query, retrieved_docs)

    return answer

# Quick Demo Test
user_question = "What is the definition of a document under Section 2?"
final_response = ask_legal_assistant(user_question)
print("\n=== FINAL ANSWER ===\n")
print(final_response)

[*] Processing query: 'What is the definition of a document under Section 2?'
[*] Retrieving relevant statutory context...
Extracted metadata filter: section_number = '2'
[*] Synthesizing context-grounded response via Ollama...

=== FINAL ANSWER ===

A document means any matter expressed or described or otherwise recorded upon any substance by means of letters, figures or marks or any other means or by more than one of those means, intended to be used, or which may be used, for the purpose of recording that matter and includes electronic and digital records. 

